In [39]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
model = ChatGroq(model = "openai/gpt-oss-20b", groq_api_key = groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EDD73181A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EDD731AD80>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
from langchain_core.messages import HumanMessage 

model.invoke([
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")
])

AIMessage(content='Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether you need help brainstorming AI projects, debugging code, or exploring the latest research, I’m here for you.', additional_kwargs={'reasoning_content': 'User says "Hello my name is shaurya , I m cheif AI Engineer". We should respond politely. We might ask how can we help.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 85, 'total_tokens': 173, 'completion_time': 0.09096012, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.004828126, 'prompt_tokens_details': None, 'queue_time': 0.043285274, 'total_time': 0.095788246}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e99e93f2ac', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07073-747e-7c32-a3c7-e49958c704e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 'output_

In [41]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer"),
    AIMessage(content="Hello shaurya, nice to meet you! How can I assist you today?"),
    HumanMessage(content="Hey Whats My name and what do I do")
])


AIMessage(content='You’re **Shaurya**, and you’re a **Chief AI Engineer**.', additional_kwargs={'reasoning_content': 'User: "Hey Whats My name and what do I do". They want to know their name and what they do. The user said earlier: "Hello my name is shaurya , I m cheif AI Engineer". So the answer: "Your name is shaurya, and you are a Chief AI Engineer." Also maybe confirm.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 121, 'total_tokens': 217, 'completion_time': 0.118410426, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.005846981, 'prompt_tokens_details': None, 'queue_time': 0.335312376, 'total_time': 0.124257407}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_3023a70d60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07073-75a9-71a1-8da3-a7138972ca9f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 121, 'output_tokens'

# Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of
inputs and outputs of the model, and store them in some datastore. Future interactions will then
load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [42]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory


store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store :
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]
        
with_message_history = RunnableWithMessageHistory(model,get_session_history)


In [43]:
config = {"configurable": {"session_id": "chat1"}}

In [44]:
response = with_message_history.invoke([
        HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")],
        config=config
)

In [45]:
response.content 

'Hello, Shaurya! 👋 It’s great to meet a Chief AI Engineer. How can I assist you today? Whether you’re looking for technical insights, brainstorming ideas, or just want to chat about AI trends, I’m here to help.'

In [46]:
res = with_message_history.invoke([
    HumanMessage(content="what is my name and what do i do ?")
], config=config)

In [47]:
res.content

'You’re Shaurya, and you’re a **Chief AI Engineer**. That means you’re leading AI strategy, overseeing development, and steering the technical direction of AI projects within your organization. If you’d like to dive into specific responsibilities or discuss any AI topics, just let me know!'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the
raw user input is just a message, which we are passing to the LLM. Letis now make that a bit more complicated.
First, let's add in a system message with some custom instructions (but still taking messages as input). Next,
we'll add in more input besides just the messages.

In [48]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful assistant , answer all the questions in the best of your abilities "),
    MessagesPlaceholder(variable_name="messages"),
])

chain = prompt | model

In [49]:
chain.invoke(
    {"messages": [HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")]}
)

AIMessage(content='Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether it’s brainstorming ideas, troubleshooting code, or exploring the latest AI trends, I’m here to help.', additional_kwargs={'reasoning_content': 'We need to respond politely, acknowledging. No constraints.'}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 105, 'total_tokens': 172, 'completion_time': 0.073920374, 'completion_tokens_details': {'reasoning_tokens': 12}, 'prompt_time': 0.005750728, 'prompt_tokens_details': None, 'queue_time': 0.332135306, 'total_time': 0.079671102}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_639a5351c8', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07073-7c20-7a60-a285-bfb9da4a0f0a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 105, 'output_tokens': 67, 'total_tokens': 172, 'output_token_details': {'reaso

In [50]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [51]:
config = {"configurable": {"session_id": "chat2"}}
response = with_message_history.invoke([
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")],
    config=config
)
response 

AIMessage(content='Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether it’s brainstorming ideas, troubleshooting a project, exploring new AI trends, or anything else, just let me know!', additional_kwargs={'reasoning_content': "We need to respond as helpful assistant. User says Hello, name is shaurya, I'm chief AI Engineer. Likely they want a greeting, maybe ask what they need. Should respond politely, maybe ask how can help."}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 105, 'total_tokens': 209, 'completion_time': 0.110936738, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.005068655, 'prompt_tokens_details': None, 'queue_time': 0.385524017, 'total_time': 0.116005393}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8b41efc9a3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07073-7e0c-7282-aa5f-d061c195b

In [52]:
response.content

'Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether it’s brainstorming ideas, troubleshooting a project, exploring new AI trends, or anything else, just let me know!'

In [53]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable": {"session_id": "chat2"}}

In [54]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")]},
    config=config
)
response.content

'Nice to meet you again, Shaurya! 👋 As the Chief AI Engineer, you’re probably juggling a lot of exciting projects. What can I help you with today? Whether it’s a technical deep‑dive, strategy advice, or just bouncing ideas around, I’m here to assist.'